In [ ]:
```json
{
  "nbformat": 4,
  "nbformat_minor": 2,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.8.10"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# ODI to Databricks Migration — SIL_InventoryProductDimension (W_INVENTORY_PRODUCT_D)\n",
        "\n",
        "**Source:** ODI Session — SILOS_SIL_INVENTORYPRODUCTDIMENSION\n",
        "\n",
        "**Target Table:** `workspace.prxbi_dw.w_inventory_product_d`\n",
        "\n",
        "**Description:** Incremental load of Inventory Product Dimension. Loads from staging DS table with multiple dimension lookups (product category, business location, internal org, product, user), performs change detection via OUTER strategy, then MERGE (update existing / insert new) into the target dimension table.\n",
        "\n",
        "**ODI KM:** IKM BIAPPS Oracle Incremental Update — Detection Strategy: OUTER, Dim ETL: Y, ROW_WID present: Y, DELETE_FLG present: Y, Auto Correction: Y"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Cell 1 — Create ETL Parameter Widgets (SCEN_TASK_NO {1} params)\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"WH_DATASOURCE_NUM_ID\", \"\")\n",
        "dbutils.widgets.text(\"ETL_USAGE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"3260538\")\n",
        "dbutils.widgets.text(\"EXECUTION_ID\", \"\")\n",
        "dbutils.widgets.text(\"IS_INCREMENTAL\", \"Y\")\n",
        "dbutils.widgets.text(\"PRUNE_DAYS\", \"0\")\n",
        "dbutils.widgets.text(\"LOW_DATE\", \"1900-01-01 00:00:00\")\n",
        "dbutils.widgets.text(\"SOURCE_CODE\", \"\")\n",
        "dbutils.widgets.text(\"TARGET_CODE\", \"\")"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {1}: Check ETL Load Dates for committed run\n",
        "SELECT (CASE\n",
        "    WHEN COUNT(*) > 0 THEN 'Y'\n",
        "    ELSE 'N'\n",
        "END) AS etl_load_check\n",
        "FROM workspace.prxbi_dw.w_etl_load_dates\n",
        "WHERE PACKAGE_NAME = 'SILOS_SIL_INVENTORYPRODUCTDIMENSION'\n",
        "AND (DATASOURCE_NUM_ID = ${DATASOURCE_NUM_ID}\n",
        "     OR DATASOURCE_NUM_ID = ${WH_DATASOURCE_NUM_ID})\n",
        "AND ETL_USAGE_CODE = '${ETL_USAGE_CODE}'\n",
        "AND COMMITTED = '1';"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- Create temporary view for LOW_DATE parameter\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_low_date AS\n",
        "SELECT to_timestamp('${LOW_DATE}', 'yyyy-MM-dd HH:mm:ss') AS low_date;"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "# Display ETL parameter values\n",
        "display(spark.sql(\"\"\"\n",
        "SELECT\n",
        "    '${DATASOURCE_NUM_ID}' AS DATASOURCE_NUM_ID,\n",
        "    '${WH_DATASOURCE_NUM_ID}' AS WH_DATASOURCE_NUM_ID,\n",
        "    '${ETL_USAGE_CODE}' AS ETL_USAGE_CODE,\n",
        "    '${ETL_PROC_WID}' AS ETL_PROC_WID,\n",
        "    '${ODI_SESS_NO}' AS ODI_SESS_NO,\n",
        "    '${IS_INCREMENTAL}' AS IS_INCREMENTAL,\n",
        "    '${PRUNE_DAYS}' AS PRUNE_DAYS,\n",
        "    '${LOW_DATE}' AS LOW_DATE,\n",
        "    '${SOURCE_CODE}' AS SOURCE_CODE,\n",
        "    '${TARGET_CODE}' AS TARGET_CODE\n",
        "\"\"\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## SCEN_TASK_NO {2} — Pre-step: Update Inventory Product Category WIDs"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {2}: MERGE to update inv_prod_cat1 and inv_prod_cat1_wid on w_inventory_product_d\n",
        "MERGE INTO workspace.prxbi_dw.w_inventory_product_d AS T\n",
        "USING (\n",
        "    SELECT DISTINCT\n",
        "        x.integration_id,\n",
        "        x.inv_prod_cat1,\n",
        "        y.inv_prod_cat1_wid\n",
        "    FROM\n",
        "        (\n",
        "            SELECT\n",
        "                b.integration_id,\n",
        "                a.integration_id AS inv_prod_cat1\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp a,\n",
        "                workspace.prxbi_dw.w_inventory_product_d b\n",
        "            WHERE\n",
        "                CONCAT(a.inventory_item_id, '~', a.organization_id) = b.integration_id\n",
        "                AND a.integration_id <> b.inv_prod_cat1\n",
        "        ) x,\n",
        "        (\n",
        "            SELECT\n",
        "                p.integration_id,\n",
        "                q.row_wid AS inv_prod_cat1_wid\n",
        "            FROM\n",
        "                workspace.prxbi_dw.w_ora_invitem_category_tmp p,\n",
        "                workspace.prxbi_dw.w_prod_cat_dh q\n",
        "            WHERE\n",
        "                q.integration_id = p.integration_id\n",
        "        ) y\n",
        "    WHERE\n",
        "        x.inv_prod_cat1 = y.integration_id\n",
        ") AS S\n",
        "ON T.integration_id = S.integration_id\n",
        "WHEN MATCHED THEN UPDATE SET\n",
        "    T.inv_prod_cat1 = S.inv_prod_cat1,\n",
        "    T.inv_prod_cat1_wid = S.inv_prod_cat1_wid;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Error Table\n",
        "\n",
        "SCEN_TASK_NO {60}–{70}: Create error table for logging"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {60}: Drop error table if exists\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.e_3260538_1;"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {70}: Create error table\n",
        "CREATE TABLE workspace.prxbi_dw.e_3260538_1\n",
        "(\n",
        "    ORA_ERR_NUMBER       DOUBLE,\n",
        "    ORA_ERR_MESG         STRING,\n",
        "    ORA_ERR_ROWID        STRING,\n",
        "    ORA_ERR_OPTYP        STRING,\n",
        "    ORA_ERR_TAG          STRING,\n",
        "    IND_UPDATE           STRING,\n",
        "    DIAGNOSTIC_ROWID     STRING,\n",
        "    ERROR_TYPE_IND       STRING,\n",
        "    AUTOCORRECT_IND      STRING DEFAULT 'N',\n",
        "    AUTOCORRECT_CODE     STRING,\n",
        "    AUTOCORRECT_DESC     STRING,\n",
        "    COMMITTED            STRING DEFAULT '0',\n",
        "    ROW_WID              STRING,\n",
        "    PRODUCT_WID          STRING,\n",
        "    INVENTORY_ORG_WID    STRING,\n",
        "    PLANT_LOC_WID        STRING,\n",
        "    PRODUCT_NUM          STRING,\n",
        "    ABC_IND              STRING,\n",
        "    PLANNER_CODE         STRING,\n",
        "    PROCUREMENT_TYPE_CODE STRING,\n",
        "    SPC_PROC_TYPE_CODE   STRING,\n",
        "    BUYER_CODE           STRING,\n",
        "    BUYER_NAME           STRING,\n",
        "    COMMODITY_CODE       STRING,\n",
        "    COMMODITY_UOM_CODE   STRING,\n",
        "    PROFIT_CENTER_NUM    STRING,\n",
        "    REORDER_POINT        STRING,\n",
        "    SAFETY_STOCK_LEVEL   STRING,\n",
        "    MIN_LOT_SIZE         STRING,\n",
        "    MAX_LOT_SIZE         STRING,\n",
        "    FIXED_LOT_SIZE       STRING,\n",
        "    MAX_STOCK_LEVEL      STRING,\n",
        "    LOT_ORDERING_COST    STRING,\n",
        "    MRP_TIME_FENCE       STRING,\n",
        "    EXT_PROCURE_TIME     STRING,\n",
        "    INTERNAL_MFG_TIME    STRING,\n",
        "    MAX_STORAGE_DAYS     STRING,\n",
        "    MRP_PROFILE_CODE     STRING,\n",
        "    MRP_TYPE_CODE        STRING,\n",
        "    MRP_GRP_CODE         STRING,\n",
        "    LOT_SIZE_CODE        STRING,\n",
        "    BACKFLUSH_IND        STRING,\n",
        "    QA_INSPECT_IND       STRING,\n",
        "    REPETITIVE_MFG_IND   STRING,\n",
        "    BULK_ITEM_IND        STRING,\n",
        "    FORECAST_PERIOD      STRING,\n",
        "    MFG_UOM_CODE         STRING,\n",
        "    ISSUE_UOM_CODE       STRING,\n",
        "    MANUFACTURING_PLACE  STRING,\n",
        "    LOADING_TYPE_CODE    STRING,\n",
        "    INT_STORE_LOC_CODE   STRING,\n",
        "    EXT_STORE_LOC_CODE   STRING,\n",
        "    ACTIVE_FLG           STRING,\n",
        "    CREATED_BY_WID       STRING,\n",
        "    CHANGED_BY_WID       STRING,\n",
        "    CREATED_ON_DT        STRING,\n",
        "    CHANGED_ON_DT        STRING,\n",
        "    AUX1_CHANGED_ON_DT   STRING,\n",
        "    AUX2_CHANGED_ON_DT   STRING,\n",
        "    AUX3_CHANGED_ON_DT   STRING,\n",
        "    AUX4_CHANGED_ON_DT   STRING,\n",
        "    SRC_EFF_FROM_DT      STRING,\n",
        "    SRC_EFF_TO_DT        STRING,\n",
        "    EFFECTIVE_FROM_DT    STRING,\n",
        "    EFFECTIVE_TO_DT      STRING,\n",
        "    CURRENT_FLG          STRING,\n",
        "    W_INSERT_DT          STRING,\n",
        "    W_UPDATE_DT          STRING,\n",
        "    DATASOURCE_NUM_ID    STRING,\n",
        "    ETL_PROC_WID         STRING,\n",
        "    INTEGRATION_ID       STRING,\n",
        "    TENANT_ID            STRING,\n",
        "    X_CUSTOM             STRING,\n",
        "    INV_PROD_CAT1        STRING,\n",
        "    INV_PROD_CAT2        STRING,\n",
        "    INV_PROD_CAT3        STRING,\n",
        "    INV_PROD_CAT4        STRING,\n",
        "    INV_PROD_CAT5        STRING,\n",
        "    INV_PROD_CAT6        STRING,\n",
        "    INV_PROD_CAT7        STRING,\n",
        "    INV_PROD_CAT8        STRING,\n",
        "    INV_PROD_CAT9        STRING,\n",
        "    INV_PROD_CAT10       STRING,\n",
        "    INV_PROD_CAT1_WID    STRING,\n",
        "    INV_PROD_CAT2_WID    STRING,\n",
        "    INV_PROD_CAT3_WID    STRING,\n",
        "    INV_PROD_CAT4_WID    STRING,\n",
        "    INV_PROD_CAT5_WID    STRING,\n",
        "    INV_PROD_CAT6_WID    STRING,\n",
        "    INV_PROD_CAT7_WID    STRING,\n",
        "    INV_PROD_CAT8_WID    STRING,\n",
        "    INV_PROD_CAT9_WID    STRING,\n",
        "    INV_PROD_CAT10_WID   STRING,\n",
        "    INVOICEABLE_ITEM_FLAG STRING,\n",
        "    INVOICE_ENABLED_FLAG STRING,\n",
        "    PRIMARY_UOM_CODE     STRING,\n",
        "    C_PRIMARY_UOM_CODE   STRING,\n",
        "    UNSPSC_CODE          STRING,\n",
        "    UNSPSC_INV_PROD_CAT_WID STRING,\n",
        "    COMMODITY_NAME       STRING,\n",
        "    COMMODITY_UOM_NAME   STRING,\n",
        "    EXT_STORE_LOC_NAME   STRING,\n",
        "    INT_STORE_LOC_NAME   STRING,\n",
        "    ISSUE_UOM_NAME       STRING,\n",
        "    LOADING_TYPE_NAME    STRING,\n",
        "    LOT_SIZE_NAME        STRING,\n",
        "    MFG_UOM_NAME         STRING,\n",
        "    MRP_GRP_NAME         STRING,\n",
        "    MRP_PROFILE_NAME     STRING,\n",
        "    MRP_TYPE_NAME        STRING,\n",
        "    PLANNER_NAME         STRING,\n",
        "    PRIMARY_UOM_NAME     STRING,\n",
        "    PROCUREMENT_TYPE_NAME STRING,\n",
        "    PROFIT_CENTER_NAME   STRING,\n",
        "    SPC_PROC_TYPE_NAME   STRING,\n",
        "    STATUS_CODE          STRING,\n",
        "    W_STATUS_CODE        STRING,\n",
        "    PRODUCT_TYPE_CODE    STRING,\n",
        "    MAKE_BUY_IND         STRING,\n",
        "    FIXED_LEAD_TIME      STRING,\n",
        "    VARIABLE_LEAD_TIME   STRING,\n",
        "    CUMULATIVE_TOTAL_LEAD_TIME STRING,\n",
        "    POSTPROCESSING_LEAD_TIME STRING,\n",
        "    PREPROCESSING_LEAD_TIME STRING,\n",
        "    PROCESS_QUALITY_ENABLED_FLG STRING,\n",
        "    X_PRICE_SEQUENCE     STRING,\n",
        "    X_ORGANIZATION_NAME  STRING,\n",
        "    X_PRODUCT_DESC       STRING,\n",
        "    X_UOM_DESC           STRING,\n",
        "    X_INV_ITEM_FLG       STRING,\n",
        "    X_STOCK_ITEM_FLG     STRING,\n",
        "    X_TRANS_FLG          STRING,\n",
        "    X_REV_FLG            STRING,\n",
        "    X_COST_FLG           STRING,\n",
        "    X_GCOA_ACCT          STRING,\n",
        "    X_GCOA_PROD          STRING,\n",
        "    X_TAX_CAT            STRING,\n",
        "    ORGANIZATION_ID      STRING,\n",
        "    X_GCOA_LOC_ACCT      STRING,\n",
        "    DELETE_FLG           STRING\n",
        ") USING DELTA;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Flow Table\n",
        "\n",
        "SCEN_TASK_NO {110}–{130}: Create and populate flow table I$_3260538_1"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {110}: Drop flow table\n",
        "DROP TABLE IF EXISTS workspace.prxbi_dw.i_3260538_1_flow;"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {120}: Create flow table\n",
        "CREATE TABLE workspace.prxbi_dw.i_3260538_1_flow\n",
        "(\n",
        "    SRC_EFF_FROM_DT              TIMESTAMP,\n",
        "    DATASOURCE_NUM_ID            BIGINT,\n",
        "    INTEGRATION_ID               STRING,\n",
        "    ROW_WID                      DOUBLE,\n",
        "    PRODUCT_WID                  DOUBLE,\n",
        "    INVENTORY_ORG_WID            DOUBLE,\n",
        "    PLANT_LOC_WID                DOUBLE,\n",
        "    PRODUCT_NUM                  STRING,\n",
        "    ABC_IND                      STRING,\n",
        "    PLANNER_CODE                 STRING,\n",
        "    PROCUREMENT_TYPE_CODE        STRING,\n",
        "    SPC_PROC_TYPE_CODE           STRING,\n",
        "    BUYER_CODE                   STRING,\n",
        "    BUYER_NAME                   STRING,\n",
        "    COMMODITY_CODE               STRING,\n",
        "    COMMODITY_UOM_CODE           STRING,\n",
        "    PROFIT_CENTER_NUM            STRING,\n",
        "    REORDER_POINT                DOUBLE,\n",
        "    SAFETY_STOCK_LEVEL           DOUBLE,\n",
        "    MIN_LOT_SIZE                 DOUBLE,\n",
        "    MAX_LOT_SIZE                 DOUBLE,\n",
        "    FIXED_LOT_SIZE               DOUBLE,\n",
        "    MAX_STOCK_LEVEL              DOUBLE,\n",
        "    LOT_ORDERING_COST            DOUBLE,\n",
        "    MRP_TIME_FENCE               DOUBLE,\n",
        "    EXT_PROCURE_TIME             DOUBLE,\n",
        "    INTERNAL_MFG_TIME            DOUBLE,\n",
        "    MAX_STORAGE_DAYS             DOUBLE,\n",
        "    MRP_PROFILE_CODE             STRING,\n",
        "    MRP_TYPE_CODE                STRING,\n",
        "    MRP_GRP_CODE                 STRING,\n",
        "    LOT_SIZE_CODE                STRING,\n",
        "    BACKFLUSH_IND                STRING,\n",
        "    QA_INSPECT_IND               STRING,\n",
        "    REPETITIVE_MFG_IND           STRING,\n",
        "    BULK_ITEM_IND                STRING,\n",
        "    FORECAST_PERIOD              STRING,\n",
        "    MFG_UOM_CODE                 STRING,\n",
        "    ISSUE_UOM_CODE               STRING,\n",
        "    MANUFACTURING_PLACE          STRING,\n",
        "    LOADING_TYPE_CODE            STRING,\n",
        "    INT_STORE_LOC_CODE           STRING,\n",
        "    EXT_STORE_LOC_CODE           STRING,\n",
        "    ACTIVE_FLG                   STRING,\n",
        "    CREATED_BY_WID               DOUBLE,\n",
        "    CHANGED_BY_WID               DOUBLE,\n",
        "    CREATED_ON_DT                TIMESTAMP,\n",
        "    CHANGED_ON_DT                TIMESTAMP,\n",
        "    AUX1_CHANGED_ON_DT           TIMESTAMP,\n",
        "    AUX2_CHANGED_ON_DT           TIMESTAMP,\n",
        "    AUX3_CHANGED_ON_DT           TIMESTAMP,\n",
        "    AUX4_CHANGED_ON_DT           TIMESTAMP,\n",
        "    SRC_EFF_TO_DT                TIMESTAMP,\n",
        "    EFFECTIVE_FROM_DT            TIMESTAMP,\n",
        "    EFFECTIVE_TO_DT              TIMESTAMP,\n",
        "    DELETE_FLG                   STRING,\n",
        "    CURRENT_FLG                  STRING,\n",
        "    W_INSERT_DT                  TIMESTAMP,\n",
        "    W_UPDATE_DT                  TIMESTAMP,\n",
        "    ETL_PROC_WID                 DOUBLE,\n",
        "    TENANT_ID                    STRING,\n",
        "    X_CUSTOM                     STRING,\n",
        "    INV_PROD_CAT1                STRING,\n",
        "    INV_PROD_CAT2                STRING,\n",
        "    INV_PROD_CAT3                STRING,\n",
        "    INV_PROD_CAT4                STRING,\n",
        "    INV_PROD_CAT5                STRING,\n",
        "    INV_PROD_CAT6                STRING,\n",
        "    INV_PROD_CAT7                STRING,\n",
        "    INV_PROD_CAT8                STRING,\n",
        "    INV_PROD_CAT9                STRING,\n",
        "    INV_PROD_CAT10               STRING,\n",
        "    INV_PROD_CAT1_WID            DOUBLE,\n",
        "    INV_PROD_CAT2_WID            DOUBLE,\n",
        "    INV_PROD_CAT3_WID            DOUBLE,\n",
        "    INV_PROD_CAT4_WID            DOUBLE,\n",
        "    INV_PROD_CAT5_WID            DOUBLE,\n",
        "    INV_PROD_CAT6_WID            DOUBLE,\n",
        "    INV_PROD_CAT7_WID            DOUBLE,\n",
        "    INV_PROD_CAT8_WID            DOUBLE,\n",
        "    INV_PROD_CAT9_WID            DOUBLE,\n",
        "    INV_PROD_CAT10_WID           DOUBLE,\n",
        "    INVOICEABLE_ITEM_FLAG        STRING,\n",
        "    INVOICE_ENABLED_FLAG         STRING,\n",
        "    PRIMARY_UOM_CODE             STRING,\n",
        "    C_PRIMARY_UOM_CODE           STRING,\n",
        "    UNSPSC_CODE                  STRING,\n",
        "    UNSPSC_INV_PROD_CAT_WID      DOUBLE,\n",
        "    COMMODITY_NAME               STRING,\n",
        "    COMMODITY_UOM_NAME           STRING,\n",
        "    EXT_STORE_LOC_NAME           STRING,\n",
        "    INT_STORE_LOC_NAME           STRING,\n",
        "    ISSUE_UOM_NAME               STRING,\n",
        "    LOADING_TYPE_NAME            STRING,\n",
        "    LOT_SIZE_NAME                STRING,\n",
        "    MFG_UOM_NAME                 STRING,\n",
        "    MRP_GRP_NAME                 STRING,\n",
        "    MRP_PROFILE_NAME             STRING,\n",
        "    MRP_TYPE_NAME                STRING,\n",
        "    PLANNER_NAME                 STRING,\n",
        "    PRIMARY_UOM_NAME             STRING,\n",
        "    PROCUREMENT_TYPE_NAME        STRING,\n",
        "    PROFIT_CENTER_NAME           STRING,\n",
        "    SPC_PROC_TYPE_NAME           STRING,\n",
        "    STATUS_CODE                  STRING,\n",
        "    W_STATUS_CODE                STRING,\n",
        "    PRODUCT_TYPE_CODE            STRING,\n",
        "    MAKE_BUY_IND                 STRING,\n",
        "    FIXED_LEAD_TIME              DOUBLE,\n",
        "    VARIABLE_LEAD_TIME           DOUBLE,\n",
        "    CUMULATIVE_TOTAL_LEAD_TIME   DOUBLE,\n",
        "    POSTPROCESSING_LEAD_TIME     DOUBLE,\n",
        "    PREPROCESSING_LEAD_TIME      DOUBLE,\n",
        "    PROCESS_QUALITY_ENABLED_FLG  STRING,\n",
        "    X_PRICE_SEQUENCE             STRING,\n",
        "    X_ORGANIZATION_NAME          STRING,\n",
        "    X_PRODUCT_DESC               STRING,\n",
        "    X_UOM_DESC                   STRING,\n",
        "    X_INV_ITEM_FLG               STRING,\n",
        "    X_STOCK_ITEM_FLG             STRING,\n",
        "    X_TRANS_FLG                  STRING,\n",
        "    X_REV_FLG                    STRING,\n",
        "    X_COST_FLG                   STRING,\n",
        "    X_GCOA_ACCT                  STRING,\n",
        "    X_GCOA_PROD                  STRING,\n",
        "    X_TAX_CAT                    STRING,\n",
        "    ORGANIZATION_ID              STRING,\n",
        "    X_GCOA_LOC_ACCT              STRING,\n",
        "    IND_UPDATE                   STRING\n",
        ") USING DELTA;"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Insert into Flow Table\n",
        "\n",
        "SCEN_TASK_NO {130}: Detection Strategy OUTER — insert into flow table with dimension lookups and change detection"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO {130}: Insert into flow table with OUTER detection strategy\n",
        "-- Converted: NVL->COALESCE, TO_DATE->to_timestamp, ||->CONCAT, ROWID->target join null check,\n",
        "--            removed /*+ append */, Oracle schemas -> workspace.prxbi_dw\n",
        "INSERT INTO workspace.prxbi_dw.i_3260538_1_flow\n",
        "(\n",
        "    PRODUCT_WID,\n",
        "    INVENTORY_ORG_WID,\n",
        "    PLANT_LOC_WID,\n",
        "    PRODUCT_NUM,\n",
        "    ABC_IND,\n",
        "    PLANNER_CODE,\n",
        "    PROCUREMENT_TYPE_CODE,\n",
        "    SPC_PROC_TYPE_CODE,\n",
        "    BUYER_CODE,\n",
        "    BUYER_NAME,\n",
        "    COMMODITY_CODE,\n",
        "    COMMODITY_UOM_CODE,\n",
        "    PROFIT_CENTER_NUM,\n",
        "    REORDER